In [ ]:
import networkx as nx
import pandas as pd
import numpy as np
import community as community_louvain
from sklearn.ensemble import IsolationForest
from collections import Counter, defaultdict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# Loading the CSV files 
train_transac = pd.read_csv("Dataset/train_transaction.csv")
test_transac = pd.read_csv("Dataset/test_transaction.csv")
train_id = pd.read_csv("Dataset/train_identity.csv")
test_id = pd.read_csv("Dataset/train_identity.csv")

# 1. Merge and Split Data Chronologically
full_train_df = pd.merge(train_transac, train_id, on='TransactionID', how='left')
full_train_df = full_train_df.sort_values('TransactionDT').reset_index(drop=True)

split_index = int(len(full_train_df) * 0.8)
df_train = full_train_df.iloc[:split_index].copy()
df_val = full_train_df.iloc[split_index:].copy()

# 2. Pre-calculate entity frequencies
print("Calculating entity frequencies for edge weighting...")
entity_frequencies = Counter()

for _, row in df_train.iterrows():
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            entity_frequencies[entity_node] += 1

# 3. Initialize the Knowledge Graph
G = nx.Graph()

# 4. Build KG with Hyper-Hub Penalized Edge Weights
hub_threshold = len(df_train) * 0.01 

for _, row in df_train.iterrows():
    tx_id = row['TransactionID']
    
    G.add_node(
        tx_id, 
        node_type='Transaction',
        amount=row.get('TransactionAmt', 0),
        is_fraud=row.get('isFraud', 0),
        timestamp=row.get('TransactionDT', 0),
        id_01=row.get('id_01', 0),
        id_02=row.get('id_02', 0)
    )
    
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            
            freq = entity_frequencies[entity_node]
            
            if freq > hub_threshold:
                weight = 0.01 / freq  # Hyper-hub penalty
            else:
                weight = 1.0 / (1.0 + np.log(freq))
            
            G.add_node(entity_node, node_type=col, frequency=freq)
            G.add_edge(tx_id, entity_node, relationship=col, weight=weight)

# 5. Extract Superior Unsupervised Community-Driven Topology Metrics & Print Diagnostics
print("Computing ring-optimized topological features...")
pagerank = nx.pagerank(G, alpha=0.85, weight='weight')
partition = community_louvain.best_partition(G, weight='weight')
nx.set_node_attributes(G, partition, 'community_id')

# --- UNSUPERVISED COMMUNITY PARTITION DIAGNOSTICS ---
community_sizes = Counter(partition.values())
print(f"\n--- Community Partition Diagnostics ---")
print(f"Total unique communities detected: {len(community_sizes)}")
print(f"Largest community size: {max(community_sizes.values())} nodes")
print(f"Smallest community size: {min(community_sizes.values())} nodes")
print(f"Median community size: {np.median(list(community_sizes.values())):.1f} nodes")

# Compute purely unsupervised community properties (e.g., average PageRank and size per community)
comm_pagerank_sums = defaultdict(float)
comm_node_counts = Counter()
for node, comm_id in partition.items():
    comm_node_counts[comm_id] += 1
    comm_pagerank_sums[comm_id] += pagerank.get(node, 0.0)

comm_avg_pagerank = {comm: comm_pagerank_sums[comm] / comm_node_counts[comm] for comm in comm_node_counts}
comm_size_dict = dict(community_sizes)

# 6. Extract Training Feature Matrix
graph_features = []
for node, data in G.nodes(data=True):
    if data.get('node_type') == 'Transaction':
        neighbors = list(G.neighbors(node))
        
        neighbor_communities = [
            G.nodes[n].get('community_id', -1) 
            for n in neighbors if 'community_id' in G.nodes[n]
        ]
        assigned_community = max(set(neighbor_communities), key=neighbor_communities.count) if neighbor_communities else -1
        
        weighted_deg = sum(G[node][n].get('weight', 1.0) for n in neighbors)
        
        graph_features.append({
            'TransactionID': node,
            'isFraud': data.get('is_fraud', 0),
            'TransactionAmt': data.get('amount', 0),
            'id_01': data.get('id_01', 0),
            'id_02': data.get('id_02', 0),
            'graph_degree': G.degree(node),
            'weighted_degree': weighted_deg,
            'pagerank': pagerank.get(node, 0),
            'assigned_community': assigned_community
        })

df_features = pd.DataFrame(graph_features)

# Map unsupervised community structural features instead of label-based rates
df_features['community_size'] = df_features['assigned_community'].map(comm_size_dict).fillna(1)
df_features['community_avg_pagerank'] = df_features['assigned_community'].map(comm_avg_pagerank).fillna(0)

# 7. Train Isolation Forest with Unsupervised Topological Features
feature_cols = [
    'TransactionAmt', 
    'id_01', 
    'id_02',
    'graph_degree', 
    'weighted_degree',
    'pagerank',
    'community_size',
    'community_avg_pagerank'
]

X_train = df_features[feature_cols].fillna(0)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

iso_forest = IsolationForest(
    n_estimators=150,
    contamination=0.03,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_train_scaled)


FileNotFoundError: [Errno 2] No such file or directory: 'Dataset/train_transaction.csv'

In [ ]:
# ------------------------------------ VALIDATION EVALUATION ------------------------------------
print("Extracting validation graph features...")
val_graph_features = []
for _, row in df_val.iterrows():
    tx_id = row['TransactionID']
    
    G.add_node(
        tx_id, 
        node_type='ValTransaction',
        amount=row.get('TransactionAmt', 0),
        is_fraud=row.get('isFraud', 0),
        timestamp=row.get('TransactionDT', 0),
        id_01=row.get('id_01', 0),
        id_02=row.get('id_02', 0)
    )
    
    for col in ['card1', 'card2', 'card3', 'card4', 'card5', 'card6', 
                'DeviceInfo', 'DeviceType', 'id_31', 'id_30', 'P_emaildomain']:
        if col in row and pd.notna(row[col]):
            val = int(row[col]) if col in ['card1', 'card2', 'card3', 'card5'] else row[col]
            entity_node = f"{col}_{val}"
            
            freq = entity_frequencies.get(entity_node, 1)
            weight = 0.01 / freq if freq > hub_threshold else 1.0 / (1.0 + np.log(freq))
            
            if not G.has_node(entity_node):
                G.add_node(entity_node, node_type=col, frequency=freq)
            G.add_edge(tx_id, entity_node, relationship=col, weight=weight)

pagerank_val = nx.pagerank(G, alpha=0.85, weight='weight')

for node, data in G.nodes(data=True):
    if data.get('node_type') == 'ValTransaction':
        neighbors = list(G.neighbors(node))
        
        neighbor_communities = [
            partition.get(n, -1) for n in neighbors if n in partition
        ]
        assigned_community = max(set(neighbor_communities), key=neighbor_communities.count) if neighbor_communities else -1
        
        comm_size = comm_size_dict.get(assigned_community, 1)
        comm_avg_pr = comm_avg_pagerank.get(assigned_community, 0)
        
        weighted_deg = sum(G[node][n].get('weight', 1.0) for n in neighbors)
        
        val_graph_features.append({
            'TransactionID': node,
            'isFraud': data.get('is_fraud', 0),
            'TransactionAmt': data.get('amount', 0),
            'id_01': data.get('id_01', 0),
            'id_02': data.get('id_02', 0),
            'graph_degree': G.degree(node),
            'weighted_degree': weighted_deg,
            'pagerank': pagerank_val.get(node, 0),
            'community_size': comm_size,
            'community_avg_pagerank': comm_avg_pr
        })

df_val_features = pd.DataFrame(val_graph_features)

# Score Validation Set
X_val = df_val_features[feature_cols].fillna(0)
X_val_scaled = scaler.transform(X_val)

df_val_features['anomaly_score'] = iso_forest.decision_function(X_val_scaled)
df_val_features['predicted_risk'] = -df_val_features['anomaly_score']

# Evaluate Performance
auc_score = roc_auc_score(df_val_features['isFraud'], df_val_features['predicted_risk'])
print(f"\nUnseen Validation ROC-AUC (Hub-Penalized Unsupervised Community KG): {auc_score:.4f}")

Extracting validation graph features...

Unseen Validation ROC-AUC (Hub-Penalized Unsupervised Community KG): 0.7148


# Export for the Streamlit app (Graph Explorer + Community Explorer + dynamic reporting)

The cells below:
1. Normalize `ValTransaction` nodes to `Transaction` (the train/val distinction was only needed to validate the Isolation Forest).
2. Compute the initial `risk_score` / `risk_class` (no reports yet) following the PDF's schema (section 10), using the formulas shared in `kg_risk.py` — the same module used by the Streamlit app, so the notebook and the app never compute risk in two different ways.
3. Save the graph (`knowledge_graph.pkl`) and the scored transactions table (`scored_validation_transactions.csv`) into the app's folder, i.e. the artifacts that `fraud.py` loads.

Requires `kg_risk.py` to be in the same folder as this notebook (or on the `PYTHONPATH`).


In [ ]:
import pickle
import kg_risk as kr

# ------------------------------------------------------------------
# 1. Normalize validation nodes to 'Transaction' (with a separate flag
#    to know which ones were in the validation split, in case it's
#    needed later)
# ------------------------------------------------------------------
for node, data in G.nodes(data=True):
    if data.get('node_type') == 'ValTransaction':
        G.nodes[node]['node_type'] = 'Transaction'
        G.nodes[node]['is_validation'] = True

df_val_features = df_val_features.rename(columns={'assigned_community': 'cluster_id'})

# ------------------------------------------------------------------
# 2. Anomaly evidence in [0,1] (min-max on predicted_risk, only for the
#    scored transactions, i.e. the validation ones)
# ------------------------------------------------------------------
pr_min, pr_max = df_val_features['predicted_risk'].min(), df_val_features['predicted_risk'].max()
df_val_features['anomaly_evidence'] = (
    (df_val_features['predicted_risk'] - pr_min) / (pr_max - pr_min)
    if pr_max > pr_min else 0.0
)

# Baseline neighbour risk = average anomaly evidence of the other scored
# transactions in the same community (relational proxy, no reports exist yet)
community_anomaly_mean = df_val_features.groupby('cluster_id')['anomaly_evidence'].transform('mean')
df_val_features['neighbour_risk_baseline'] = community_anomaly_mean.fillna(df_val_features['anomaly_evidence'])

df_val_features['risk_score'] = df_val_features.apply(
    lambda r: kr.transaction_baseline_risk(r['anomaly_evidence'], r['neighbour_risk_baseline']),
    axis=1,
)
df_val_features['risk_class'] = df_val_features['risk_score'].map(kr.risk_class)
df_val_features['report_evidence'] = 0.0
df_val_features['num_reports'] = 0

# ------------------------------------------------------------------
# 3. Write the same values back as attributes on the Transaction nodes
#    of the graph
# ------------------------------------------------------------------
tx_attrs = df_val_features.set_index('TransactionID')[
    ['anomaly_score', 'anomaly_evidence', 'cluster_id', 'risk_score', 'risk_class',
     'report_evidence', 'num_reports']
].to_dict(orient='index')

for tx_id, attrs in tx_attrs.items():
    if G.has_node(tx_id):
        G.nodes[tx_id].update(attrs)

# Training transactions were never scored by the Isolation Forest in this
# notebook: give them a cluster_id (from the Louvain partition) and a
# neutral risk, so they still show up in the Graph/Community Explorer
# without polluting the metrics.
for node, data in G.nodes(data=True):
    if data.get('node_type') == 'Transaction' and 'risk_score' not in data:
        G.nodes[node]['cluster_id'] = partition.get(node, -1)
        G.nodes[node]['anomaly_score'] = 0.0
        G.nodes[node]['anomaly_evidence'] = 0.0
        G.nodes[node]['risk_score'] = 0.0
        G.nodes[node]['risk_class'] = kr.RISK_LOW
        G.nodes[node]['report_evidence'] = 0.0
        G.nodes[node]['num_reports'] = 0

# ------------------------------------------------------------------
# 4. Initial risk score for entities (card/device/email/...): average
#    risk of the connected transactions, no anomaly score of their own
# ------------------------------------------------------------------
for node, data in G.nodes(data=True):
    if data.get('node_type') in kr.ENTITY_COLS:
        neighbour_scores = [
            G.nodes[n]['risk_score']
            for n in G.neighbors(node)
            if G.nodes[n].get('node_type') == 'Transaction' and 'risk_score' in G.nodes[n]
        ]
        neighbour_risk = sum(neighbour_scores) / len(neighbour_scores) if neighbour_scores else 0.0
        G.nodes[node]['risk_score'] = kr.entity_risk(0.0, neighbour_risk)
        G.nodes[node]['risk_class'] = kr.risk_class(G.nodes[node]['risk_score'])
        G.nodes[node]['report_evidence'] = 0.0
        G.nodes[node]['num_reports'] = 0

print(f"Transactions with risk_score: {sum(1 for _, d in G.nodes(data=True) if d.get('node_type') == 'Transaction')}")
print(f"Total entity nodes: {sum(1 for _, d in G.nodes(data=True) if d.get('node_type') in kr.ENTITY_COLS)}")

# ------------------------------------------------------------------
# 5. Save the artifacts that fraud.py expects
# ------------------------------------------------------------------
with open("knowledge_graph.pkl", "wb") as f:
    pickle.dump(G, f, protocol=pickle.HIGHEST_PROTOCOL)

export_cols = [
    'TransactionID', 'TransactionAmt', 'isFraud', 'graph_degree', 'weighted_degree',
    'pagerank', 'cluster_id', 'community_size', 'community_avg_pagerank',
    'anomaly_score', 'anomaly_evidence', 'risk_score', 'risk_class',
    'report_evidence', 'num_reports',
]
df_val_features[export_cols].to_csv("scored_validation_transactions.csv", index=False)

print("Saved: knowledge_graph.pkl, scored_validation_transactions.csv")
